In [6]:
from google import genai
from google.genai import types
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.embeddings import Embeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.stores import InMemoryStore
from langchain_core.documents import Document
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_google_genai  import ChatGoogleGenerativeAI
from langchain_community.retrievers import BM25Retriever
import requests
from pathlib import Path
import os
from dotenv import load_dotenv
import uuid
import hashlib


load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=30)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)

In [7]:
class GeminiEmbeddings(Embeddings):
    def __init__(self, api_key, model="gemini-embedding-2"):
        self.client = genai.Client(api_key=api_key)
        self.model = model

    def embed_documents(self, texts):

        embeddings = []
        batch_size = 50
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            batch_contents = [
                    types.Content(
                        parts=[types.Part(text=text)],
                        role="user",
                    )
                    for text in batch
                    ]
                    
            resp = self.client.models.embed_content(
                model=self.model,
                contents=batch_contents
            )

            embeddings.extend(e.values for e in resp.embeddings)

        return embeddings

    def embed_query(self, text):
        resp = self.client.models.embed_content(
            model=self.model,
            contents=[text]
        )
        return resp.embeddings[0].values

gemini_embeddings = GeminiEmbeddings(api_key=api_key)



In [37]:
docs_list = []
child_docs_list = []
parent_docs_list = []
RAG_NAMESPACE = uuid.UUID('7d5a5286-6df7-4404-b97c-e0938f381c15')
pdf_dir = Path(r"C:\Users\cmanw\OneDrive\Documents\AI-Projects\Project_1\PDF_FOLDER")
for pdf_path in pdf_dir.glob("*.pdf"):
    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()

    
    with open(pdf_path, "rb") as f:
        file_hash = hashlib.sha256(f.read()).hexdigest()

    document_id = file_hash
    
    for doc in docs:
        doc.metadata["pdf_name"] = pdf_path.name
        doc.metadata["document_id"] = document_id
    docs_list.extend(docs)

    
    parent_docs = parent_splitter.split_documents(docs)
    for parent_doc in parent_docs:
        parent_id = uuid.uuid5(RAG_NAMESPACE, parent_doc.page_content)
        parent_doc.metadata["parent_id"] = parent_id
    parent_docs_list.extend(parent_docs)
   
    
    for i, parent_doc in enumerate(parent_docs, start=1):
        child_docs = child_splitter.split_documents([parent_doc])
        for child_doc in child_docs:
            child_id = uuid.uuid5(RAG_NAMESPACE, child_doc.page_content)
            child_doc.metadata["child_id"] = child_id
        child_docs_list.extend(child_docs)

child_texts = [doc.page_content for doc in child_docs_list]
embedded_contents = gemini_embeddings.embed_documents(child_texts)




# vectorstore = Chroma.from_documents(embedding_functions=gemini_embeddings, persist_directory="chroma_db2")

# vectorstore.add_documents(child_docs_list)

In [38]:
len(child_texts)

81

In [39]:
len(embedded_contents)

81

In [11]:
len(parent_docs)

33

In [42]:
import psycopg2 

conn = psycopg2.connect(
    database="vectordb3",
    user="postgres",
    password="newpassword",  # <-- Change this from "password"
    host="127.0.0.1",
    port=5432
)
cursor = conn.cursor()


In [45]:
for doc in docs_list:
    file_hash = doc.metadata["document_id"]
    pdf_name = doc.metadata["pdf_name"]
    cursor.execute(
        "INSERT INTO documents (file_hash, pdf_name) VALUES (%s, %s) ON CONFLICT (file_hash) DO NOTHING",
        (file_hash, pdf_name)
    )
conn.commit()



In [47]:
for parent_doc in parent_docs_list:
    parent_docs_id = parent_doc.metadata["parent_id"] 
    hash_id = parent_doc.metadata["document_id"] 
    parent_pages = parent_doc.metadata["page"]  #mark where the page number is coming from
    parent_texts = parent_doc.page_content 
    cursor.execute(
        "INSERT INTO parent_chunks (parent_id, file_hash, page, parent_texts) VALUES (%s,%s,%s,%s) ON CONFLICT (parent_id) DO NOTHING", 
        (str(parent_docs_id), hash_id, parent_pages, parent_texts)
        )
conn.commit()

In [49]:
for child_doc, embedded_content in zip(child_docs_list, embedded_contents):
    child_id = child_doc.metadata["child_id"]
    child_parent_id = child_doc.metadata["parent_id"]
    child_text = child_doc.page_content
    embedding = embedded_content
    cursor.execute(
        "INSERT INTO  child_chunks (child_id, parent_id, child_text, embeddings) VALUES (%s,%s,%s,%s) ON CONFLICT (child_id) DO NOTHING", 
        (str(child_id), str(child_parent_id), child_text, embedding)
    )
conn.commit()

In [50]:
query = "What is the agenti ai about?"
embed_query = gemini_embeddings.embed_query(query)

string_embed = str(embed_query)
print(string_embed)

[-0.009234365, 0.016348159, -0.0024183795, -0.00018624196, -0.005949042, -0.006578937, -0.025904931, 0.0034174235, -0.015850699, -0.045914512, -0.016546924, 0.0013616461, 0.0052804737, -0.028182993, 0.0055961353, -0.02927566, 0.03631701, -0.0015293994, 0.01032234, 0.007776206, -0.015587472, -0.0104964785, 0.015733162, 0.0151843, 0.012665299, 0.029678011, -0.016220475, 0.003506201, -0.02194332, 0.11227695, -0.00019339236, -0.014644962, 0.02272475, 0.009153233, 0.02518818, 0.0109827025, -0.024296118, -0.01653012, 0.026940871, 0.0016086851, 0.0057923975, 0.022436284, 0.0072443015, 0.0044419486, -0.014630817, -0.00641804, -0.007350134, 0.0074262638, 0.0037726173, 0.0018477967, 0.015285693, 0.0122910235, -0.0021974104, -0.019924328, 0.008548471, -0.023971766, 0.028904127, -0.0049979757, -0.019952172, 0.014201176, -0.0072169974, -0.0065204166, 0.0060184128, 0.012498926, 0.0007647579, -0.01505766, 0.00027923883, -0.024530623, 0.025043964, -0.029310523, -0.01354869, 0.017919783, -0.0015445345,

In [ ]:
sql_query = """ 
WITH ranked_child_chunks AS ( 
    SELECT parent_id, (embeddings <=> %s::vector ) AS distance FROM child_chunks 
    ORDER BY embeddings <=> %s::vector ASC
    LIMIT 20
),

deduplicated_parent_ids AS (
    SELECT  parent_id , MIN(distance) as best_distance from ranked_child_chunks
    GROUP BY parent_id
)

SELECT p.parent_id, p.parent_texts, p.page FROM parent_chunks p
JOIN deduplicated_parent_ids d on p.parent_id = d.parent_id
WHERE length(p.parent_texts) > 100 
ORDER BY d.best_distance ASC
LIMIT 5 
"""
cursor.execute(sql_query, (string_embed, string_embed))

In [52]:
results = cursor.fetchall()
results

[('dae85634-34f3-51fe-936d-ddceee258528',
  "What is an \nagent?\nWhile conventional software enables users to streamline and automate workflows, agents are able \nto perform the same workflows on the users’ behalf with a high degree of independence.\nAgents are systems that independently accomplish tasks on your behalf.\nA workflow is a sequence of steps that must be executed to meet the user’s goal, whether that's \nresolving a customer service issue, booking a restaurant reservation, committing a code change, \u2028\nor generating a report.\nApplications that integrate LLMs but don’t use them to control workflow execution—think simple \nchatbots, single-turn LLMs, or sentiment classifiers—are not agents.\nMore concretely, an agent possesses core characteristics that allow it to act reliably and \nconsistently on behalf of a user:\n01 It leverages an LLM to manage workflow execution and make decisions. It recognizes \nwhen a workflow is complete and can proactively correct its action

In [43]:
conn.rollback()

In [ ]:


# for text, emb in zip(texts, embedded_contents):
#     cursor.execute("INSERT INTO items (text, embedding) VALUES (%s, %s)", (text, emb.tolist()))

# conn.commit()
# result = cursor.fetchall()
# result

In [ ]:
cursor = conn.cursor()

In [ ]:
batch_size = 5
embeddings = []


for i in range(0, len(corpus), batch_size):
    batch = corpus[i:i + batch_size]

    # Convert only this batch into Content objects
    batch_contents = [
        types.Content(
            parts=[types.Part(text=text)],
            role="user",
        )
        for text in batch
    ]

    print(f"Batch size: {len(batch)}")

    resp = client.models.embed_content(
        model="gemini-embedding-2",
        contents=batch_contents,
    )

    print(f"Returned embeddings: {len(resp.embeddings)}")

    embeddings.extend(e.values for e in resp.embeddings)

print(f"Total embeddings: {len(embeddings)}")

In [ ]:
len(embeddings)

In [ ]:
from sentence_transformers import SentenceTransformer

model =  SentenceTransformer('all-MiniLM-L6-v2')


In [ ]:
corpus = [
    "The quick lantern flickered beneath azure arches.",
    "Curious pigeons whispered secrets on the rooftop.",
    "A lone violin echoed through the empty hall.",
    "Midnight rain drummed softly on the cobblestones.",
    "She folded the paper boat with trembling fingers.",
    "Autumn leaves danced in a lazy spiral.",
    "The old clock chimed thirteen times at dawn.",
    "Velvet shadows stretched across the cobbler's lane.",
    "A distant ocean breeze carried salt and stories.",
]

In [ ]:
embedding = model.encode(corpus)
embedding.shape

In [ ]:
import psycopg2 

In [ ]:
conn = psycopg2.connect(
    database= "vectordb2",
    user= "postgres",
    password= "newpassword",
    host="127.0.0.4",
    port=5432
)
cursor = conn.cursor()
# for text, emb in zip(corpus, embedding):
#     cursor.execute("INSERT INTO items (text, embedding) VALUES (%s, %s)", (text, emb.tolist()))

# conn.commit()
# result = cursor.fetchall()
# result

In [ ]:
cursor = conn.cursor()

In [ ]:
for text, emb in zip(corpus, embedding):
    cursor.execute("INSERT INTO items (text, embedding) VALUES (%s, %s)", (text, emb.tolist()))

conn.commit()

In [ ]:
query_emb = embedding[0].tolist()
cursor.execute(f"SELECT * FROM items ORDER BY embedding <-> '{query_emb}' LIMIT 2")

In [ ]:
result = cursor.fetchall()
result

In [ ]:
result